# SetFit Fine-tune — Boilerplate Classifier (Stage 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidavi111/bpclassifier/blob/main/notebooks/colab/setfit_finetune.ipynb)

**Runtime:** GPU (T4). Set via *Runtime → Change runtime type → T4 GPU*.

**What this does:**
1. Pulls `colab-data:v0` from W&B (train + val splits with sentence text)
2. Fine-tunes `BAAI/bge-small-en-v1.5` with SetFit contrastive learning, 5-fold CV
3. Trains a final model on the full train split
4. Pushes `model-setfit` and `oof-setfit` artifacts back to W&B

**Before running:** Add `WANDB_API_KEY` to Colab Secrets (key icon in left panel).

**IMPORTANT — run Cell 1 first and alone.** It checks the setfit version and auto-restarts
the kernel if an upgrade is needed. After the restart, run all remaining cells normally.

In [ ]:
# Cell 1: ensure setfit>=1.1.0 is installed, auto-restart if an upgrade was needed.
# Run this cell FIRST and ALONE. After a kernel restart, run from Cell 2 onwards.
import sys
import subprocess

def _setfit_version():
    try:
        import setfit
        return setfit.__version__
    except Exception:
        return '0.0.0'

from packaging.version import Version

_current = _setfit_version()
print(f'setfit version detected: {_current}')

if Version(_current) < Version('1.1.0'):
    print('setfit is too old — installing setfit>=1.1.0 and restarting kernel...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'setfit>=1.1.0', 'wandb', 'pyarrow', 'scikit-learn', 'datasets'],
        check=True,
    )
    print('Install complete. Restarting runtime — continue from Cell 2 after restart.')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print(f'setfit {_current} is already >=1.1.0 — installing remaining packages...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'wandb', 'pyarrow', 'scikit-learn', 'datasets'],
        check=True,
    )
    print('All packages ready. Proceed to Cell 2.')

In [ ]:
# Cell 2: verify version, load W&B key, initialise run.
import setfit
from packaging.version import Version
assert Version(setfit.__version__) >= Version('1.1.0'), (
    f'setfit {setfit.__version__} is still too old — re-run Cell 1 and allow the restart.'
)
print(f'setfit {setfit.__version__} confirmed.')

import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception:
    print('Colab Secrets not found — will prompt for login.')

import wandb
wandb.login()

WANDB_ENTITY       = 'david-avichzer-hebrew-university-of-jerusalem'
WANDB_PROJECT      = 'Boilerplate_Classifier'
BASE_MODEL         = 'BAAI/bge-small-en-v1.5'
N_FOLDS            = 5
SEED               = 42
CONTRASTIVE_EPOCHS = 1
BATCH_SIZE         = 16

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name='stage5-setfit',
    job_type='train',
    tags=['stage:5-zoo', 'purpose:train', 'model:setfit', 'split:oof'],
    config={
        'model': BASE_MODEL,
        'n_folds': N_FOLDS,
        'contrastive_epochs': CONTRASTIVE_EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
    },
)
print(f'Run: {run.url}')

In [ ]:
# Cell 3: pull colab-data:v0 from W&B.
import numpy as np
import pandas as pd
from pathlib import Path

art = run.use_artifact('colab-data:v0')
art_dir = Path(art.download())

train_df = pd.read_parquet(art_dir / 'train_with_text.parquet')
val_df   = pd.read_parquet(art_dir / 'val_with_text.parquet')
print(f'Train: {len(train_df)} rows | Val: {len(val_df)} rows')
print(f"Label distribution (train): {train_df['gold_label'].value_counts().to_dict()}")

In [ ]:
# Cell 4: import SetFit and define helpers.
import time
from datasets import Dataset as HFDataset
from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArgs
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score

print('SetFit imported successfully.')

y_train = (train_df['gold_label'] == 'substantive').astype(int).values
groups  = (train_df['ticker'] + '_' + train_df['quarter'].astype(str)).values


def make_model():
    return SetFitModel.from_pretrained(
        BASE_MODEL,
        labels=['boilerplate', 'substantive'],
    )


def get_probs(model, texts):
    """Return p(substantive) as float array."""
    raw = model.predict_proba(list(texts))
    arr = raw.numpy() if hasattr(raw, 'numpy') else np.array(raw, dtype=float)
    return np.clip(arr[:, 1], 0.0, 1.0)


def make_hf_dataset(texts, labels):
    return HFDataset.from_dict({'text': list(texts), 'label': [int(l) for l in labels]})

In [ ]:
# Cell 5: 5-fold OOF cross-validation.
skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs    = np.zeros(len(train_df), dtype=float)
fold_metrics = []

for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(train_df, y_train, groups)):
    print(f'\n=== Fold {fold_idx} ===')
    tr_ds     = make_hf_dataset(train_df['text'].iloc[tr_idx], y_train[tr_idx])
    vl_texts  = train_df['text'].iloc[vl_idx]
    vl_labels = y_train[vl_idx]

    model = make_model()
    args  = SetFitArgs(
        num_epochs=CONTRASTIVE_EPOCHS,
        batch_size=BATCH_SIZE,
        seed=SEED,
        report_to='none',
    )
    trainer = SetFitTrainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()

    probs = get_probs(model, vl_texts)
    oof_probs[vl_idx] = probs

    preds = (probs >= 0.5).astype(int)
    fm = {
        'fold': fold_idx,
        'macro_f1': float(f1_score(vl_labels, preds, average='macro')),
        'substantive_recall': float(recall_score(vl_labels, preds, pos_label=1)),
    }
    fold_metrics.append(fm)
    print(f"Fold {fold_idx}: F1={fm['macro_f1']:.4f}  recall={fm['substantive_recall']:.4f}")
    del model, trainer

mean_f1     = float(np.mean([f['macro_f1'] for f in fold_metrics]))
mean_recall = float(np.mean([f['substantive_recall'] for f in fold_metrics]))
print(f'\nOOF macro-F1={mean_f1:.4f}  sub-recall={mean_recall:.4f}')

In [ ]:
# Cell 6: train final model on full train set, evaluate on val.
y_val = (val_df['gold_label'] == 'substantive').astype(int).values

full_tr_ds = make_hf_dataset(train_df['text'], y_train)

t0 = time.perf_counter()
final_model = make_model()
args = SetFitArgs(
    num_epochs=CONTRASTIVE_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    report_to='none',
)
trainer = SetFitTrainer(model=final_model, args=args, train_dataset=full_tr_ds)
trainer.train()
train_sec = time.perf_counter() - t0

t1 = time.perf_counter()
val_probs = get_probs(final_model, val_df['text'])
infer_sec = time.perf_counter() - t1
throughput = len(val_df) / infer_sec

val_preds  = (val_probs >= 0.5).astype(int)
val_f1     = float(f1_score(y_val, val_preds, average='macro'))
val_recall = float(recall_score(y_val, val_preds, pos_label=1))
print(f'Val F1={val_f1:.4f}  recall={val_recall:.4f}  throughput={throughput:.0f} sps')

MODEL_SAVE_DIR = '/content/setfit_model'
final_model.save_pretrained(MODEL_SAVE_DIR)
print(f'Model saved to {MODEL_SAVE_DIR}')

oof_path      = Path('/content/oof_train_setfit.parquet')
val_pred_path = Path('/content/pred_val_setfit.parquet')
pd.DataFrame({'sentence_id': train_df['sentence_id'].values, 'prob_substantive': oof_probs}).to_parquet(oof_path, index=False)
pd.DataFrame({'sentence_id': val_df['sentence_id'].values, 'prob_substantive': val_probs}).to_parquet(val_pred_path, index=False)
print(f'OOF: {len(oof_probs)} rows | Val preds: {len(val_probs)} rows')

In [ ]:
# Cell 7: upload model-setfit and oof-setfit artifacts to W&B, finish run.
wandb.log({
    'oof_macro_f1': mean_f1,
    'oof_substantive_recall': mean_recall,
    'val_macro_f1': val_f1,
    'val_substantive_recall': val_recall,
    'training_time_sec': train_sec,
    'infer_throughput_sps': throughput,
    **{f"fold_{f['fold']}_macro_f1": f['macro_f1'] for f in fold_metrics},
})

model_art = wandb.Artifact(
    'model-setfit',
    type='model',
    description='BAAI/bge-small-en-v1.5 fine-tuned with SetFit for boilerplate/substantive classification',
    metadata={'val_macro_f1': val_f1, 'base_model': BASE_MODEL},
)
model_art.add_dir(MODEL_SAVE_DIR)
run.log_artifact(model_art)

oof_art = wandb.Artifact(
    'oof-setfit',
    type='predictions',
    description='5-fold OOF probs + val preds from SetFit',
    metadata={'oof_macro_f1': mean_f1, 'val_macro_f1': val_f1},
)
oof_art.add_file(str(oof_path), name='oof_train_setfit.parquet')
oof_art.add_file(str(val_pred_path), name='pred_val_setfit.parquet')
run.log_artifact(oof_art)

run.finish()
print('W&B run finished.')

In [ ]:
# Cell 8: verify artifacts were pushed successfully.
import wandb
api = wandb.Api()
art = api.artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/oof-setfit:latest')
files = [f.name for f in art.files()]
print(f'Confirmed oof-setfit files: {files}')
assert 'oof_train_setfit.parquet' in files, 'OOF parquet missing from artifact!'
assert 'pred_val_setfit.parquet' in files, 'Val-pred parquet missing from artifact!'
print('DONE — push to W&B successful')